# AI Usage Protocol Notebook

Use this notebook to document relevant AI/LLM interactions for one milestone.

## Motivation and Relevance

This protocol is intended to support the documentation of AI usage in accordance with current university and NRW-related guidelines requiring transparency of AI-assisted work processes. In particular, students must disclose relevant uses of AI tools, including revisions, translations, text generation, image generation, analysis support, structuring support, or other AI-assisted scientific activities.

This protocol is not intended as a raw chat dump and not as a collection of successful AI interactions only.

Students must document all milestone-relevant AI interactions that substantially influenced scientific reasoning, understanding, decision-making, literature evaluation, argumentation, modeling, implementation, or interpretation — independent of whether the AI output was ultimately accepted, revised, or rejected.

Relevant interactions therefore include:

* accepted outputs,
* partially used outputs,
* rejected outputs,
* hallucinated or misleading outputs,
* exploratory interactions,
* and interactions that triggered validation or correction activities.

The protocol focuses on documenting scientific reasoning and validation processes, not only final integration into the submitted work.

For iterative or multi-step workflows, students should document the interaction as a sequence of connected reasoning episodes. This includes relevant follow-up prompts, refinements, manual modifications, validation loops, and corrections whenever they substantially influenced the scientific reasoning or outcome of the milestone work.
## Workflow

1. Execute the notebook initialization cell and fill in the general metadata fields for the milestone work that become visible afterwards.
2. For each scientifically relevant AI-supported reasoning episode, click **Add interaction**.
3. Document the relevant prompts, follow-up refinements, manual modifications, validation activities, evaluations, and reflections related to this reasoning episode.
4. Repeat this process for all relevant AI-supported reasoning episodes that influenced the milestone work — including accepted, revised, rejected, exploratory, or corrective usages.
5. Click **Save protocol JSON** to generate the machine-readable protocol file and the corresponding BibTeX entry (`.bib`).
6. Submit the completed notebook (`.ipynb`), the generated protocol (`.json`), and the generated BibTeX file (`.bib`) together with the milestone artifact.


In [ ]:
import json
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError as e:
    raise ImportError("ipywidgets is required. Install with: pip install ipywidgets") from e

PROTOCOL_VERSION = "0.2"

verification_levels = [
    "none",
    "plausibility-check",
    "secondary-source-check",
    "primary-source-check",
    "methodological-check",
    "empirical-check",
    "multi-source-validation"
]

evaluation_statuses = [
    "accepted",
    "partially-accepted",
    "rejected",
    "revised",
    "unresolved"
]

usefulness_levels = [
    "low",
    "medium",
    "high"
]

usage_types = [
    "idea-generation",
    "literature-search-support",
    "summarization",
    "terminology-clarification",
    "comparison-structuring",
    "argument-critique",
    "text-revision",
    "translation",
    "code-generation",
    "verification-support",
    "modeling-support",
    "requirements engineering-support",
    "other"
]

metadata = {
    "protocol_version": PROTOCOL_VERSION,
    "student_id": "",
    "course": "",
    "milestone": "",
    "topic": "",
    "entries": []
}

student_id = widgets.Text(description="Student ID:", placeholder="anonymous or matriculation-compatible ID")
course = widgets.Text(description="Course:", placeholder="Course/module name")
milestone = widgets.Text(description="Milestone:", placeholder="e.g. literature-analysis")
topic = widgets.Text(description="Topic:", placeholder="Your seminar/lab topic")

metadata_box = widgets.VBox([
    widgets.HTML("<h3>Protocol Metadata</h3>"),
    student_id,
    course,
    milestone,
    topic
])

entries_box = widgets.VBox([])
status_output = widgets.Output()

def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")

def make_interaction_widget(parent_id=None):
    entry_id = f"e-{uuid.uuid4().hex[:8]}"

    objective = widgets.Textarea(
        description="Objective:",
        placeholder="What scientific task did you try to solve?",
        layout=widgets.Layout(width="100%", height="70px")
    )

    ai_tool = widgets.Text(description="AI tool:", placeholder="e.g. ChatGPT, Claude, Perplexity")
    ai_model = widgets.Text(description="Model:", placeholder="e.g. GPT-5.5, unknown")

    prompt = widgets.Textarea(
        description="Prompt:",
        placeholder="Paste the relevant prompt here.",
        layout=widgets.Layout(width="100%", height="120px")
    )

    ai_output_summary = widgets.Textarea(
        description="AI output summary:",
        placeholder="Summarize the AI output. Do not paste long raw outputs unless required.",
        layout=widgets.Layout(width="100%", height="100px")
    )

    verification_level = widgets.RadioButtons(
        options=verification_levels,
        description="Verification level:",
        value="plausibility-check"
    )

    verification_actions = widgets.Textarea(
        description="Verification actions:",
        placeholder="What exactly did you check, and against which sources/evidence?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    evaluation_status = widgets.RadioButtons(
        options=evaluation_statuses,
        description="Status:",
        value="partially-accepted"
    )

    usefulness = widgets.RadioButtons(
        options=usefulness_levels,
        description="Usefulness:",
        value="medium"
    )

    issues = widgets.Textarea(
        description="Issues:",
        placeholder="List hallucinations, overgeneralizations, missing assumptions, wrong citations, etc.",
        layout=widgets.Layout(width="100%", height="90px")
    )

    used_in_work = widgets.Checkbox(description="Used in submitted work", value=False)
    section = widgets.Text(description="Section:", placeholder="e.g. Section 2.3")
    usage_type = widgets.Dropdown(options=usage_types, description="Usage type:", value="other")
    direct_text_reused = widgets.Checkbox(description="Direct text reused", value=False)

    reflection = widgets.Textarea(
        description="Reflection:",
        placeholder="What did you learn? How did this affect your understanding or scientific decision?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    remove_button = widgets.Button(description="Remove interaction", button_style="danger")

    box = widgets.VBox([
        widgets.HTML(f"<hr><h3>Interaction {entry_id}</h3>"),
        objective,
        widgets.HBox([ai_tool, ai_model]),
        prompt,
        ai_output_summary,
        widgets.HTML("<b>Verification</b>"),
        verification_level,
        verification_actions,
        widgets.HTML("<b>Evaluation</b>"),
        evaluation_status,
        usefulness,
        issues,
        widgets.HTML("<b>Integration</b>"),
        used_in_work,
        section,
        usage_type,
        direct_text_reused,
        widgets.HTML("<b>Reflection</b>"),
        reflection,
        remove_button
    ])

    box.entry_id = entry_id
    box.parent_id = parent_id
    box.fields = {
        "objective": objective,
        "ai_tool": ai_tool,
        "ai_model": ai_model,
        "prompt": prompt,
        "ai_output_summary": ai_output_summary,
        "verification_level": verification_level,
        "verification_actions": verification_actions,
        "evaluation_status": evaluation_status,
        "usefulness": usefulness,
        "issues": issues,
        "used_in_work": used_in_work,
        "section": section,
        "usage_type": usage_type,
        "direct_text_reused": direct_text_reused,
        "reflection": reflection
    }

    def remove_entry(_):
        children = list(entries_box.children)
        if box in children:
            children.remove(box)
            entries_box.children = tuple(children)

    remove_button.on_click(remove_entry)
    return box

def add_interaction(_):
    entries_box.children = tuple(list(entries_box.children) + [make_interaction_widget()])

def serialize_entry(box):
    f = box.fields

    verification_actions = [
        line.strip() for line in f["verification_actions"].value.splitlines()
        if line.strip()
    ]
    issues = [
        line.strip() for line in f["issues"].value.splitlines()
        if line.strip()
    ]

    return {
        "id": box.entry_id,
        "timestamp": now_iso(),
        "parent_id": box.parent_id,
        "objective": f["objective"].value.strip(),
        "ai_tool": {
            "name": f["ai_tool"].value.strip(),
            "model": f["ai_model"].value.strip()
        },
        "prompt": f["prompt"].value.strip(),
        "ai_output_summary": f["ai_output_summary"].value.strip(),
        "verification": {
            "actions": verification_actions,
            "level": f["verification_level"].value
        },
        "evaluation": {
            "status": f["evaluation_status"].value,
            "issues": issues,
            "usefulness": f["usefulness"].value
        },
        "integration": {
            "used_in_work": bool(f["used_in_work"].value),
            "section": f["section"].value.strip(),
            "usage_type": f["usage_type"].value,
            "direct_text_reused": bool(f["direct_text_reused"].value)
        },
        "reflection": f["reflection"].value.strip()
    }


def safe_bibtex_key(text):
    """Create a BibTeX-safe key component."""
    text = (text or "ai-usage-protocol").lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text or "ai-usage-protocol"

def escape_bibtex(value):
    """Escape a minimal set of characters for BibTeX fields."""
    value = str(value or "")
    return (
        value.replace("\\", "\\textbackslash{}")
             .replace("{", "\\{")
             .replace("}", "\\}")
             .replace("&", "\\&")
             .replace("%", "\\%")
             .replace("$", "\\$")
             .replace("#", "\\#")
             .replace("_", "\\_")
    )

def protocol_to_bibtex(protocol, json_filename=None):
    """
    Convert an AI usage protocol dictionary into a BibTeX entry.

    The generated entry cites the submitted AI usage protocol as a
    documented part of the student's scientific workflow. It does not
    replace the required disclosure of specific AI tools in the protocol.
    """
    exported_at = protocol.get("exported_at") or now_iso()
    year = exported_at[:4]
    student = protocol.get("student_id") or "anonymous"
    course_name = protocol.get("course") or "Course"
    milestone_name = protocol.get("milestone") or "Milestone"
    topic_name = protocol.get("topic") or "AI-assisted scientific work"
    version = protocol.get("protocol_version") or PROTOCOL_VERSION
    entries_count = len(protocol.get("entries", []))
    json_file = json_filename or "ai_usage_protocol.json"

    key_parts = [
        "ai-usage-protocol",
        safe_bibtex_key(student),
        safe_bibtex_key(milestone_name),
        year,
    ]
    bib_key = "-".join([part for part in key_parts if part])

    title = f"AI Usage Protocol for {milestone_name}: {topic_name}"
    note = (
        f"Machine-readable documentation of AI-assisted scientific work; "
        f"course: {course_name}; protocol version: {version}; "
        f"documented reasoning episodes: {entries_count}; "
        f"JSON file: {json_file}"
    )

    return f"""@misc{{{bib_key},
  author = {{{escape_bibtex(student)}}},
  title = {{{escape_bibtex(title)}}},
  year = {{{escape_bibtex(year)}}},
  howpublished = {{{escape_bibtex("Submitted AI usage protocol notebook and JSON file")}}},
  note = {{{escape_bibtex(note)}}}
}}"""


def save_protocol(_):
    metadata["student_id"] = student_id.value.strip()
    metadata["course"] = course.value.strip()
    metadata["milestone"] = milestone.value.strip()
    metadata["topic"] = topic.value.strip()
    metadata["exported_at"] = now_iso()
    metadata["entries"] = [serialize_entry(box) for box in entries_box.children]

    base_filename = f"ai_usage_protocol_{metadata['student_id'] or 'anonymous'}_{metadata['milestone'] or 'milestone'}"
    base_filename = base_filename.replace(" ", "_").replace("/", "-")
    json_filename = f"{base_filename}.json"
    bib_filename = f"{base_filename}.bib"

    Path(json_filename).write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    bibtex_entry = protocol_to_bibtex(metadata, json_filename=json_filename)
    Path(bib_filename).write_text(bibtex_entry + "\n", encoding="utf-8")

    with status_output:
        clear_output()
        print(f"Saved protocol JSON: {json_filename}")
        print(f"Saved BibTeX entry: {bib_filename}")
        print(f"Entries: {len(metadata['entries'])}")
        print()
        print("BibTeX entry:")
        print(bibtex_entry)

add_button = widgets.Button(description="Add interaction", button_style="primary")
save_button = widgets.Button(description="Save protocol JSON", button_style="success")

add_button.on_click(add_interaction)
save_button.on_click(save_protocol)

display(metadata_box)
display(widgets.HBox([add_button, save_button]))
display(entries_box)
display(status_output)


# ------------------------------------------------------------------
# Prefilled protocol for HLS_FINAL
# Review each entry and replace any reconstructed prompt/model detail
# with the exact original information when available.
# ------------------------------------------------------------------
student_id.value = "2219941"
course.value = "Electronic Engineering"
milestone.value = "Final Seminar Report: Literature Review, Experiment, Code Generation, Modeling, and Critical Analysis"
topic.value = "High-Level Synthesis for Multi-Core Systems"

_prefilled_entries = [{'objective': 'Understand the assigned High-Level Synthesis topic and determine the scientific scope and structure of the final five-page IEEE seminar report.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Analyze the main reference and the seminar requirements for High-Level Synthesis for Multi-Core Systems. Identify the mathematical foundations, formal problem description, scheduling algorithm, runtime and overhead analysis, C/C++ implementation, application example, RTL output, limitations, and suitable additional references.', 'ai_output_summary': 'The AI proposed a report structure covering the HLS flow, data-flow graphs, ASAP/ALAP scheduling, mobility, PE partitioning, communication-aware scheduling, resource binding, runtime analysis, C/C++ implementation, an application example, RTL generation, limitations, and references.', 'verification_level': 'multi-source-validation', 'verification_actions': "Compared the proposed structure with the lecturer's seminar requirements.\nChecked the HLS flow and terminology against Elliott, Coussy and Morawiec, Gupta et al., and De Micheli.\nChecked the scheduling discussion against Paulin and Knight and Sllame and Drabek.\nChecked the main resource-sharing claims against Casseau and Le Gal.", 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'The main reference studies timewise mutually exclusive multi-mode hardware rather than simultaneous multi-core execution.\nEarly drafts risked presenting multi-mode area results as direct multi-core benchmark results.\nThe final report therefore distinguishes clearly between multi-core parallel execution and multi-mode resource sharing.', 'used_in_work': True, 'section': 'Abstract; Sections I, II, III, VI, and References', 'usage_type': 'literature-search-support', 'direct_text_reused': False, 'reflection': 'I learned that a relevant reference can support only part of a topic. The multi-mode paper is valuable for joint scheduling, binding, and path-cost analysis, but its experimental results must not be described as simultaneous multi-core results.'}, {'objective': 'Develop the mathematical foundations and formal communication-aware multi-core HLS model used in the report.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Formally describe High-Level Synthesis for Multi-Core Systems using a data-flow graph, ASAP and ALAP bounds, mobility, partition, schedule, binding, communication delay, resource constraints, speedup, an objective function, and a communication-aware list-scheduling algorithm.', 'ai_output_summary': 'The AI formulated the DFG G=(V,E), ASAP/ALAP mobility, the mappings for partition, schedule, binding and communication, dependency and resource constraints, a weighted optimization objective, a priority function using 1/(mobility+1), a communication-aware binding cost, and formal algorithm steps.', 'verification_level': 'methodological-check', 'verification_actions': 'Checked each dependency equation for producer latency and cross-PE communication delay.\nVerified that 1/(mobility+1) remains defined when mobility is zero.\nChecked that resource constraints prevent simultaneous over-allocation.\nCompared the path-aware binding cost with the main reference.\nSeparated feasibility and correctness from optimality.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'An earlier priority expression using 1/mobility was undefined for zero-mobility critical nodes.\nThe heuristic algorithm provides a valid schedule but does not prove global optimality.\nController-area values from the main reference require wording that identifies them as multi-mode results.', 'used_in_work': True, 'section': 'Sections II and III; Equations (1)-(15); Algorithm 1', 'usage_type': 'modeling-support', 'direct_text_reused': True, 'reflection': 'I learned to distinguish a correct feasible schedule from an optimal schedule. The formal model also showed that partitioning, scheduling, binding, and communication decisions are tightly coupled.'}, {'objective': 'Implement and evaluate a small C++ list scheduler for the 12-node 2x2 matrix-multiplication data-flow graph.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Create a portable C++ list-scheduling example for a 2x2 matrix-multiplication DFG. Map ready operations to two processing elements, explain the output, compare the greedy assignment with a row-based partition, and identify any cross-PE communication.', 'ai_output_summary': 'The AI supplied the Operation structure, predecessor-completion test, greedy scheduling loop, compilation command, schedule interpretation, and a row-based partition that keeps product-to-sum dependencies local.', 'verification_level': 'empirical-check', 'verification_actions': 'Compiled and executed sched.cpp with g++.\nInspected the printed operation-to-step and operation-to-PE mapping.\nCompared every addition with the locations of its producer multiplications.\nChecked the row-based schedule against the assumed multiplication and addition latencies.\nRecalculated the ideal speedup S(2)=12/6=2.0.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'The simplified code chooses the first free PE and does not implement the complete communication-aware binding score.\nAn early written interpretation of the scheduler output did not fully match the actual output.\nThe simplified per-step functional-unit model does not fully represent every possible multi-cycle initiation interval.', 'used_in_work': True, 'section': 'Section IV-A, Section IV-B, Listing 1, and Table II', 'usage_type': 'code-generation', 'direct_text_reused': True, 'reflection': 'Running the implementation showed why generated explanations must be checked against actual output. The greedy scheduler can create avoidable communication even when the arithmetic workload appears balanced.'}, {'objective': 'Design, run, and interpret a POSIX-thread experiment that models multiple processing elements and measures practical speedup and overhead.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Create a Linux POSIX pthread matrix-multiplication experiment in which each thread represents one PE. Use row partitioning, test K=1,2,4 workers, report the median of six runs, calculate speedup and efficiency, and explain why the measured scaling is below ideal.', 'ai_output_summary': 'The AI proposed the pthread implementation, CLOCK_MONOTONIC timing, row partitioning, six-run median reporting, the speedup and efficiency equations, and an interpretation involving synchronization, operating-system scheduling, and timer overhead.', 'verification_level': 'empirical-check', 'verification_actions': 'Compiled matrix_pthread.c with gcc -O2 -pthread.\nExecuted the experiment on the stated AMD Ryzen 7 Linux machine.\nRepeated the measurement six times and used median values.\nRecalculated S(2)=1.70, S(4)=2.31, and efficiencies of 85.0% and 57.8%.\nChecked that the report describes the experiment as a functional software model rather than cycle-accurate hardware.', 'evaluation_status': 'accepted', 'usefulness': 'high', 'issues': 'pthread_join and usleep are software analogues rather than real RTL communication and operator timing.\nThe measured overhead includes OS scheduling and timer granularity.\nThe experiment does not model memory-port contention or cycle-accurate interconnect arbitration.', 'used_in_work': True, 'section': 'Section IV-C and Table III', 'usage_type': 'verification-support', 'direct_text_reused': False, 'reflection': 'I learned that more workers can reduce runtime without producing proportional speedup. This provided experimental motivation for including communication and overhead terms in the formal objective.'}, {'objective': 'Evaluate the area effect of resource sharing using two VHDL datapath versions synthesized in Vivado.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Explain the separate and shared VHDL datapath experiment for two mutually exclusive arithmetic modes. Calculate the LUT reduction from 262 to 212, relate the result to HLS allocation and binding, and state the limitations of the comparison.', 'ai_output_summary': 'The AI explained the two arithmetic modes, the multiplexed shared multiplier, the 19.1% LUT reduction, and the relationship between operator reuse, multiplexing, controller cost, allocation, and binding.', 'verification_level': 'empirical-check', 'verification_actions': 'Checked the Vivado utilization reports for the separate and shared designs.\nRecalculated (262-212)/262 x 100 = 19.1%.\nConfirmed that both versions use the same external interface and I/O count.\nCompared the interpretation with the resource-sharing and interconnection-cost discussion in the main reference.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'The experiment uses mutually exclusive modes and therefore validates the area and binding principle, not simultaneous multi-PE execution.\nThe exact number of arithmetic units must be checked against the full VHDL, not only a short listing excerpt.\nFPGA LUT savings are not directly equivalent to ASIC standard-cell savings.', 'used_in_work': True, 'section': 'Section V-A and Table IV', 'usage_type': 'verification-support', 'direct_text_reused': True, 'reflection': 'I learned that resource sharing is beneficial only when operation lifetimes or modes do not overlap, and that multiplexing and controller overhead can reduce the theoretical area saving.'}, {'objective': "Explain how the scheduled and bound multi-core design becomes an RTL netlist and connect the report's algorithm to VHDL components.", 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Describe the RTL output of the HLS flow for the matrix example. Explain functional units, registers, multiplexers, communication buffers, FSM controllers, and provide a small VHDL entity example.', 'ai_output_summary': 'The AI described the per-PE RTL structure, the mapping of cut edges to buffers or communication elements, the relationship between schedule states and FSM control signals, and a representative shared-adder VHDL entity.', 'verification_level': 'primary-source-check', 'verification_actions': 'Compared the RTL structure with Coussy and Morawiec and the main Casseau and Le Gal reference.\nChecked that the described components correspond to scheduling and binding decisions.\nVerified that the row-based matrix partition requires no cross-PE product-to-sum buffers in the simplified example.\nChecked the VHDL entity syntax during LaTeX compilation.', 'evaluation_status': 'accepted', 'usefulness': 'high', 'issues': 'The entity skeleton is illustrative and is not a complete generated RTL design.\nA realistic design would additionally require reset behavior, complete control logic, memory interfaces, and top-level interconnection.', 'used_in_work': True, 'section': 'Section V-B', 'usage_type': 'terminology-clarification', 'direct_text_reused': True, 'reflection': 'This interaction connected the abstract schedule and binding mappings to actual hardware components. It clarified that HLS output is a controlled RTL datapath, not simply translated C syntax.'}, {'objective': 'Structure, revise, format, and technically review the final five-page IEEE report and disclose the use of AI.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Using the final presentation, main reference, experiments, code, and measured results, prepare a five-page IEEE-format report on High-Level Synthesis for Multi-Core Systems. Improve the organization and language, preserve the verified content, add limitations and an AI-use acknowledgment, and repair LaTeX compilation errors without changing the intended writing.', 'ai_output_summary': 'The AI helped organize and revise the report, generated IEEE LaTeX, integrated equations, algorithms, listings, tables, citations, limitations, and acknowledgment text, and repaired malformed LaTeX environments and invisible Unicode-character errors.', 'verification_level': 'methodological-check', 'verification_actions': 'Compared the final report with the primary references and recorded experiment outputs.\nChecked equations, scheduler assumptions, measured values, percentages, tables, and limitations.\nCompiled the LaTeX source with pdfLaTeX and inspected the PDF.\nCorrected statements that mixed multi-mode and multi-core results.\nRetained responsibility for the final wording and submission.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'Several intermediate LaTeX versions contained malformed environments and invisible U+200B characters.\nSome generated prose overstated experimental conclusions or did not match the code output.\nAdding the acknowledgment affected the page count and required layout review.', 'used_in_work': True, 'section': 'Whole report, especially structure, language revision, LaTeX formatting, limitations, and acknowledgment', 'usage_type': 'text-revision', 'direct_text_reused': True, 'reflection': 'I learned to use AI output as a draft rather than as evidence. Equations, citations, code behavior, and measurements were verified separately, and I retained full responsibility for the final report.'}, {'objective': 'Obtain an additional language and structure review of the HLS seminar report using Claude.', 'ai_tool': 'Claude', 'ai_model': 'Unknown / not recorded', 'prompt': 'Reconstructed purpose of the interaction: review the HLS seminar report for academic clarity, grammar, structure, and consistency while preserving the equations, experimental values, code behavior, and technical meaning. Replace this text with the exact original prompt if it is still available.', 'ai_output_summary': 'Claude was used as an additional prose-editing aid. Suggestions concerned wording, organization, and readability. Only selected suggestions were retained after technical comparison with the references and measured results.', 'verification_level': 'methodological-check', 'verification_actions': 'Compared retained suggestions with the equations, code, experiment logs, and cited sources.\nRejected language changes that altered technical meaning or numerical values.\nManually reviewed all retained edits.', 'evaluation_status': 'partially-accepted', 'usefulness': 'medium', 'issues': 'The exact Claude model and original prompt were not retained.\nThis is a retrospective entry and should be replaced with the exact record if available.\nLanguage suggestions were not used as scientific evidence.', 'used_in_work': True, 'section': 'Language and structure review across the report', 'usage_type': 'text-revision', 'direct_text_reused': False, 'reflection': 'Using a second AI tool showed that language suggestions vary between tools. Technical verification still had to rely on the references, source code, and experimental results.'}]

_prefilled_boxes = []
for _data in _prefilled_entries:
    _box = make_interaction_widget()
    _f = _box.fields
    _f["objective"].value = _data["objective"]
    _f["ai_tool"].value = _data["ai_tool"]
    _f["ai_model"].value = _data["ai_model"]
    _f["prompt"].value = _data["prompt"]
    _f["ai_output_summary"].value = _data["ai_output_summary"]
    _f["verification_level"].value = _data["verification_level"]
    _f["verification_actions"].value = _data["verification_actions"]
    _f["evaluation_status"].value = _data["evaluation_status"]
    _f["usefulness"].value = _data["usefulness"]
    _f["issues"].value = _data["issues"]
    _f["used_in_work"].value = _data["used_in_work"]
    _f["section"].value = _data["section"]
    _f["usage_type"].value = _data["usage_type"]
    _f["direct_text_reused"].value = _data["direct_text_reused"]
    _f["reflection"].value = _data["reflection"]
    _prefilled_boxes.append(_box)

entries_box.children = tuple(_prefilled_boxes)


## Recommended Use

For every milestone, start from a fresh copy of this notebook template.

Suggested rule for students:

- Do **not** document every trivial AI interaction.
- Do document every interaction that influenced a scientific decision, interpretation, comparison, argument, model, implementation, or submitted text.
- The most important part is not the prompt. The most important part is the **verification and evaluation**.

Suggested submission:

- completed notebook `.ipynb`
- generated protocol `.json`
- generated BibTeX file `.bib`
- milestone artifact, e.g. literature analysis, outline, model, code, or paper draft


## Controlled Vocabulary

### Verification level

- `none`: accepted without checking
- `plausibility-check`: checked only for plausibility
- `secondary-source-check`: checked against tutorials, summaries, lecture material, or secondary sources
- `primary-source-check`: checked against original papers, standards, documentation, or data
- `methodological-check`: checked the reasoning, method, assumptions, or formal correctness
- `empirical-check`: checked by running an experiment, model, tool, test, or implementation
- `multi-source-validation`: checked against multiple independent reliable sources

### Evaluation status

- `accepted`: used essentially as provided
- `partially-accepted`: used selectively
- `rejected`: discarded after evaluation
- `revised`: substantially corrected or rewritten
- `unresolved`: uncertainty remains


## Optional: Convert an Existing Protocol JSON to BibTeX

Use the following cell if you already have a generated protocol `.json` file and need to recreate the corresponding BibTeX entry.


In [3]:
# Optional JSON-to-BibTeX converter
#
# Change this filename if needed, then run the cell.
# The generated .bib file can be included in the references of the scientific work.

json_protocol_file = "ai_usage_protocol_anonymous_milestone.json"

def convert_json_protocol_to_bibtex(json_file):
    json_path = Path(json_file)
    if not json_path.exists():
        raise FileNotFoundError(f"Protocol JSON file not found: {json_file}")

    protocol = json.loads(json_path.read_text(encoding="utf-8"))
    bibtex_entry = protocol_to_bibtex(protocol, json_filename=json_path.name)
    bib_path = json_path.with_suffix(".bib")
    bib_path.write_text(bibtex_entry + "\n", encoding="utf-8")
    print(f"Saved BibTeX entry: {bib_path}")
    print()
    print(bibtex_entry)

# Uncomment the next line after setting json_protocol_file correctly:
# convert_json_protocol_to_bibtex(json_protocol_file)
